# Fullstack GPT: #5.0 ~ #5.8

- [x] Implement an LCEL chain with a memory that uses one of the memory classes we learned about.
- [x] The chain should take the title of a movie and reply with three emojis that represent the movie. (i.e "Top Gun" -> "🛩️👨‍✈️🔥". "The Godfather" -> "👨‍👨‍👦🔫🍝 ").
- [x] Provide examples to the chain using FewShotPromptTemplate or FewShotChatMessagePromptTemplate to make sure it always replies with three emojis.
- [x] To check that the memory is working ask the chain about two movies and then in another cell ask the chain to tell you what is the movie you asked about first.

In [19]:
from langchain.memory import ConversationBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain.schema.runnable import RunnablePassthrough

examples = [
    {
        "title": "Top Gun",
        "answer": "🛩️👨‍✈️🔥"
    },
    {
        "title": "The Godfather",
        "answer": "👨‍👨‍👦🔫🍝"
    }
]

def load_memory(_):
    return memory.load_memory_variables({})["history"]

def invoke_chain(title):
    result = chain.invoke({"question": title})
    memory.save_context({"input": title}, {"output": result.content})

memory = ConversationBufferMemory()
llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.3,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

example_prompt = PromptTemplate.from_template("Human: {title}\nAI: {answer}")

fewshot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Human: {question}",
    input_variables=["question"]
)

final_prompt = PromptTemplate.from_template("""
    Reply with three emojis that represent the given movie.

    {fewshot_prompt}      
    {history}
    Human: {question}
    AI:
""")

chain = RunnablePassthrough.assign(fewshot_prompt=fewshot_prompt, history=load_memory) | final_prompt | llm
invoke_chain("love actually")
invoke_chain("Rainy Day in New York")

❤️🎄💌☔🎬❤️

In [20]:
load_memory(_)

'Human: love actually\nAI: ❤️🎄💌\nHuman: Rainy Day in New York\nAI: ☔🎬❤️'

In [22]:
chain.invoke({"question": "Do not make emojis this time and tell me. What was the movie i asked about first? Except for movie in the fewshot prompt."})

The movie you asked about first was "Top Gun."

AIMessageChunk(content='The movie you asked about first was "Top Gun."')